In [ ]:
# 📋 总结报告
print("📋 强化学习PMM策略回测分析总结报告")
print("=" * 80)

print(f"\n🎯 分析概况:")
print(f"   分析时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   策略类型: 强化学习优化的PMM (专业做市商) 策略")
if 'training_pair' in config_metadata:
    print(f"   训练交易对: {config_metadata['training_pair']}")
print(f"   测试数据集: {len(available_data)} 个")
print(f"   成功回测: {len(backtest_results)} 个")

if optimal_config:
    print(f"\n🏆 强化学习优化参数:")
    print(
        f"   价差配置: {optimal_config['bid_spread']*10000:.1f}/{optimal_config['ask_spread']*10000:.1f} bps")
    print(
        f"   时间配置: 刷新{optimal_config['order_refresh_time']:.1f}s / 挂单{optimal_config['hang_order_time_limit']:.1f}s")
    print(
        f"   数量配置: {optimal_config['max_open_orders']}个订单 × {optimal_config['order_amount']}张")
    print(
        f"   风控配置: 止盈{optimal_config['take_profit_pct']*10000:.1f}bps / 止损{optimal_config['stop_loss_pct']*10000:.1f}bps")

if backtest_results:
    print(f"\n📊 回测表现摘要:")

    best_performance = None
    best_sharpe = float('-inf')

    for dataset_name, result in backtest_results.items():
        summary = result['summary']
        try:
            if hasattr(summary, 'to_pandas'):
                summary_df = summary.to_pandas()
            else:
                summary_df = summary

            row = summary_df.iloc[0] if len(summary_df) > 0 else None

            if row is not None:
                sharpe = getattr(row, 'SR', float('-inf'))
                if isinstance(sharpe, (int, float)) and sharpe > best_sharpe:
                    best_sharpe = sharpe
                    best_performance = dataset_name

                print(f"   {dataset_name}: 夏普比率 {sharpe:.4f}" if isinstance(
                    sharpe, (int, float)) else f"   {dataset_name}: 数据不可用")
        except:
            print(f"   {dataset_name}: 分析失败")

    if best_performance:
        print(f"\n🥇 最佳表现: {best_performance} (夏普比率: {best_sharpe:.4f})")

print(f"\n✅ 强化学习PMM策略分析完成!")
print(f"   所有结果已保存到 stats/ 目录")
print(f"   配置文件位于 checkpoints/ 目录")
print(f"\n🚀 可以将优化后的参数应用到实际交易中进行验证")

In [ ]:
# 💰 收益分析和策略建议
print("💰 强化学习策略收益分析和建议...")

if backtest_results and optimal_config:
    print(f"\n📊 基于强化学习优化的收益预估:")
    
    # 使用强化学习配置计算理论收益
    config = optimal_config
    
    # 计算理论日交易频率
    orders_per_minute = 60 / (config['order_refresh_time'] + config['hang_order_time_limit'] / 2)
    daily_orders = orders_per_minute * 60 * 24
    
    # 计算理论日交易量
    daily_volume = daily_orders * config['order_amount'] * 0.5  # 假设50%执行率
    
    # 费率收入（负费率做市商）
    maker_fee_rate = 0.0002  # 0.02% 费率收入
    daily_fee_income = daily_volume * maker_fee_rate
    
    print(f"\n🎯 强化学习优化策略收益预估:")
    print(f"   理论日订单数: {daily_orders:.0f}")
    print(f"   理论日交易量: {daily_volume:.0f} 张")
    print(f"   日费率收入: {daily_fee_income:.2f} USDT")
    print(f"   月收入: {daily_fee_income * 30:.0f} USDT")
    print(f"   年收入: {daily_fee_income * 365:.0f} USDT")
    
    # 基于实际回测结果的收益分析
    if backtest_results:
        print(f"\n📈 实际回测表现分析:")
        
        total_results = len(backtest_results)
        profitable_results = 0
        total_return = 0
        
        for dataset_name, result in backtest_results.items():
            summary = result['summary']
            
            # 尝试提取收益数据
            try:
                if hasattr(summary, 'to_pandas'):
                    summary_df = summary.to_pandas()
                else:
                    summary_df = summary
                
                row = summary_df.iloc[0] if len(summary_df) > 0 else None
                
                if row is not None:
                    daily_return = getattr(row, 'DailyAvgReturn', 0)
                    if isinstance(daily_return, (int, float)) and daily_return > 0:
                        profitable_results += 1
                        total_return += daily_return
                        print(f"   {dataset_name}: 日收益率 {daily_return:.6f}")
                    else:
                        print(f"   {dataset_name}: 收益数据不可用")
                        
            except Exception as e:
                print(f"   {dataset_name}: 分析失败 - {e}")
        
        if total_results > 0:
            success_rate = profitable_results / total_results * 100
            avg_return = total_return / profitable_results if profitable_results > 0 else 0
            
            print(f"\n🏆 强化学习策略综合评价:")
            print(f"   成功率: {success_rate:.1f}% ({profitable_results}/{total_results})")
            if avg_return > 0:
                print(f"   平均日收益率: {avg_return:.6f}")
                print(f"   年化收益率: {avg_return * 365 * 100:.2f}%")

    print(f"\n💡 策略建议:")
    print(f"   1. 强化学习优化的参数相比手工调参更加精细")
    print(f"   2. 价差设置: {config['bid_spread']*10000:.1f}bps 非常紧密，适合高流动性市场")
    print(f"   3. 订单刷新频率: {60/config['order_refresh_time']:.1f}次/分钟，保持市场活跃参与")
    print(f"   4. 风险控制: 止盈止损都设为 {config['take_profit_pct']*10000:.1f}bps，平衡风险收益")
    print(f"   5. 建议在真实交易前进一步验证参数在不同市场条件下的稳定性")

else:
    print("❌ 缺少回测结果或配置数据，无法进行收益分析")

In [ ]:
# 📈 可视化分析
print("📈 生成可视化图表...")

if backtest_results:
    # 为每个成功的回测生成图表
    for dataset_name, result in backtest_results.items():
        print(f"\n📊 {dataset_name} 策略表现图表:")
        try:
            # 生成策略表现图
            result['stats'].plot()
            print(f"   ✅ {dataset_name} 图表生成完成")
        except Exception as e:
            print(f"   ❌ {dataset_name} 图表生成失败: {e}")
else:
    print("❌ 没有可用的回测结果生成图表")

In [ ]:
# 📊 性能对比分析
print("📊 强化学习优化策略性能对比分析...")

if backtest_results:
    print(f"\n📈 各数据集表现对比:")
    print(f"{'='*80}")
    
    comparison_data = []
    
    for dataset_name, result in backtest_results.items():
        summary = result['summary']
        
        # 提取关键指标（适配polars DataFrame）
        if hasattr(summary, 'to_pandas'):
            summary_df = summary.to_pandas()
        else:
            summary_df = summary
        
        # 获取第一行数据
        row = summary_df.iloc[0] if len(summary_df) > 0 else None
        
        if row is not None:
            metrics = {
                '数据集': dataset_name,
                '夏普比率': getattr(row, 'SR', 'N/A'),
                'Sortino比率': getattr(row, 'Sortino', 'N/A'),
                '最大回撤': getattr(row, 'MDD', 'N/A'),
                '日收益率': getattr(row, 'DailyAvgReturn', 'N/A'),
                '日交易量': getattr(row, 'DailyTurnover', 'N/A'),
            }
            comparison_data.append(metrics)
            
            print(f"\n🎯 {dataset_name}:")
            for key, value in metrics.items():
                if key != '数据集':
                    if isinstance(value, (int, float)) and value != 'N/A':
                        if 'ratio' in key.lower() or '比率' in key:
                            print(f"   {key}: {value:.4f}")
                        elif '%' in str(value) or 'return' in key.lower() or '收益' in key:
                            print(f"   {key}: {value:.6f}")
                        else:
                            print(f"   {key}: {value:.4f}")
                    else:
                        print(f"   {key}: {value}")
    
    # 计算综合表现
    if len(comparison_data) > 1:
        print(f"\n🏆 强化学习策略综合评价:")
        print(f"   测试数据集数量: {len(comparison_data)}")
        
        # 计算平均性能（仅对数值指标）
        numeric_metrics = ['夏普比率', 'Sortino比率', '最大回撤', '日收益率', '日交易量']
        
        for metric in numeric_metrics:
            values = []
            for data in comparison_data:
                val = data.get(metric, 'N/A')
                if isinstance(val, (int, float)) and val != 'N/A':
                    values.append(val)
            
            if values:
                avg_val = np.mean(values)
                std_val = np.std(values) if len(values) > 1 else 0
                print(f"   平均{metric}: {avg_val:.4f} (±{std_val:.4f})")

else:
    print("❌ 没有成功的回测结果进行分析")

In [ ]:
# 🎯 执行强化学习优化策略回测
print("🎯 开始执行强化学习优化策略回测分析...")

backtest_results = {}

# 对每个可用数据集运行回测
for dataset_name, data_path in available_data.items():
    print(f"\n{'='*60}")
    print(f"🔍 测试数据集: {dataset_name}")
    print(f"{'='*60}")
    
    try:
        # 运行回测
        recorder = run_rl_optimized_backtest(
            data_path, 
            optimal_config, 
            f"{dataset_name}_RL优化"
        )
        
        # 分析结果
        stats, summary = analyze_backtest_results(
            recorder, 
            f"{dataset_name}_RL优化"
        )
        
        # 保存结果
        backtest_results[dataset_name] = {
            'stats': stats,
            'summary': summary,
            'recorder': recorder
        }
        
        print(f"✅ {dataset_name} 回测分析完成")
        
    except Exception as e:
        print(f"❌ {dataset_name} 回测失败: {e}")
        continue

print(f"\n🎉 所有数据集回测完成！成功完成 {len(backtest_results)} / {len(available_data)} 个测试")

In [ ]:
# 🧪 定义回测函数
def run_rl_optimized_backtest(data_path, config, test_name="回测"):
    """运行强化学习优化后的策略回测"""
    print(f"\n🚀 开始 {test_name}...")
    
    # 加载数据
    if data_path.endswith('.npz'):
        data_file = np.load(data_path)
        # 尝试不同的数据键名
        if 'data' in data_file:
            market_data = data_file['data']
        elif 'events' in data_file:
            market_data = data_file['events']
        else:
            # 使用第一个可用的键
            key = list(data_file.keys())[0]
            market_data = data_file[key]
            print(f"   使用数据键: {key}")
    
    print(f"   数据长度: {len(market_data):,} 条")
    
    # 配置交易资产
    asset = (
        BacktestAsset()
        .data([market_data])
        .linear_asset(1.0)
        .risk_adverse_queue_model()
        .no_partial_fill_exchange()
        .tick_size(0.001)      # SOL精度
        .lot_size(0.1)         # 最小交易量
        .trading_value_fee_model(-0.0002, 0.0007)  # 负费率做市商
        .last_trades_capacity(0)
    )
    
    # 创建回测引擎
    hbt = HashMapMarketDepthBacktest([asset])
    recorder = Recorder(1, 1_000_000_000)  # 1秒记录间隔
    
    # 执行强化学习优化的PMM策略
    start_time = time.time()
    _ = pmm.pmm_strategy(
        hbt, recorder.recorder,
        interval_seconds=0.1,  # 100ms间隔
        **config  # 使用强化学习优化的参数
    )
    execution_time = time.time() - start_time
    
    # 关闭回测引擎
    _ = hbt.close()
    
    print(f"   ✅ 回测完成，用时: {execution_time:.2f}秒")
    
    return recorder

def analyze_backtest_results(recorder, test_name="策略"):
    """分析回测结果"""
    print(f"\n📊 分析 {test_name} 结果...")
    
    # 保存原始数据
    stats_file = f'stats/rl_optimized_{test_name.lower().replace(" ", "_")}.npz'
    recorder.to_npz(stats_file)
    
    # 加载并分析数据
    data = np.load(stats_file)['0']
    stats = (
        LinearAssetRecord(data)
        .resample('1s')  # 1秒重采样
        .stats(book_size=10_000_000)  # 1000万USDT账户
    )
    
    # 显示统计摘要
    summary = stats.summary()
    print(f"\n📈 {test_name} 性能摘要:")
    display(summary)
    
    return stats, summary

print("✅ 回测函数定义完成")

In [ ]:
# 📊 准备回测数据
print("📊 准备回测数据...")

# 加载多个时间段的数据进行全面测试
data_files = {
    'SOLUSDT_20250629': 'data/output/solusdt_20250629.npz',
    'SOLUSDT_20250630': 'data/output/solusdt_20250630.npz',
    'ETHUSDT_20250629': 'data/output/ethusdt_20250629.npz',
}

# 检查数据文件存在性
available_data = {}
for name, path in data_files.items():
    if os.path.exists(path):
        available_data[name] = path
        print(f"✅ {name}: {path}")
    else:
        print(f"❌ {name}: {path} (不存在)")

if not available_data:
    raise FileNotFoundError("没有找到可用的数据文件")

print(f"\n📈 将使用 {len(available_data)} 个数据集进行回测分析")

In [ ]:
# 🔍 读取强化学习生成的最优策略配置
print("🔍 查找强化学习生成的配置文件...")

# 查找配置文件
config_files = glob('checkpoints/*optimal_config*.json')
if not config_files:
    # 如果没有找到配置文件，使用05中的示例最优配置
    print("⚠️ 未找到配置文件，使用强化学习训练结果中的最优配置")
    
    # 基于05训练结果的最优配置（负费率做市商）
    optimal_config = {
        "bid_spread": 0.000050,      # 0.5 bps
        "ask_spread": 0.000050,      # 0.5 bps  
        "order_refresh_time": 0.75,  # 0.75秒
        "price_deviation_pct": 0.0008,  # 0.8 bps
        "hang_order_time_limit": 15.5,  # 15.5秒
        "max_open_orders": 120,      # 120个订单
        "take_profit_pct": 0.000050, # 0.5 bps
        "stop_loss_pct": 0.000050,   # 0.5 bps
        "order_amount": 2500         # 2500张
    }
    
    config_metadata = {
        "training_pair": "SOLUSDT",
        "maker_fee_rate": -0.0002,
        "taker_fee_rate": -0.0001,
        "source": "强化学习训练结果"
    }
    
else:
    # 读取最新的配置文件
    latest_config_file = max(config_files, key=os.path.getmtime)
    print(f"✅ 找到配置文件: {latest_config_file}")
    
    with open(latest_config_file, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    
    optimal_config = config_data['optimal_config']
    config_metadata = config_data['metadata']

print(f"\n🏆 强化学习最优策略配置:")
print(f"   买入价差: {optimal_config['bid_spread']:.6f} ({optimal_config['bid_spread']*10000:.1f}bps)")
print(f"   卖出价差: {optimal_config['ask_spread']:.6f} ({optimal_config['ask_spread']*10000:.1f}bps)")
print(f"   刷新时间: {optimal_config['order_refresh_time']:.2f}s")
print(f"   价格偏差: {optimal_config['price_deviation_pct']:.6f} ({optimal_config['price_deviation_pct']*10000:.1f}bps)")
print(f"   挂单时限: {optimal_config['hang_order_time_limit']:.1f}s")
print(f"   最大挂单: {optimal_config['max_open_orders']}个")
print(f"   止盈比例: {optimal_config['take_profit_pct']:.6f} ({optimal_config['take_profit_pct']*10000:.1f}bps)")
print(f"   止损比例: {optimal_config['stop_loss_pct']:.6f} ({optimal_config['stop_loss_pct']*10000:.1f}bps)")
print(f"   订单数量: {optimal_config['order_amount']}张")

# 显示配置元数据
if 'training_pair' in config_metadata:
    print(f"\n📊 配置信息:")
    print(f"   训练交易对: {config_metadata['training_pair']}")
    if 'maker_fee_rate' in config_metadata:
        print(f"   Maker费率: {config_metadata['maker_fee_rate']*100:.3f}%")
    if 'average_reward' in config_metadata:
        print(f"   训练平均奖励: {config_metadata['average_reward']:.4f}")
        print(f"   训练成功率: {config_metadata['success_rate']:.1f}%")

In [ ]:
# 环境设置和依赖导入
import numpy as np
import json
import os
import time
from glob import glob
from hftbacktest import (
    BacktestAsset,
    HashMapMarketDepthBacktest,
    Recorder
)
from hftbacktest.stats import LinearAssetRecord
import lib.pmm as pmm

print("📊 回测分析环境初始化完成")

# 基于强化学习的PMM策略回测分析

本notebook将使用05_reinforcement_learning.ipynb生成的最优策略配置进行详细回测分析，评估强化学习优化后的策略表现。

## 分析目标

- **策略验证**: 验证强化学习生成的最优参数在真实市场数据上的表现
- **性能对比**: 与手工调参策略进行对比分析
- **风险评估**: 分析最大回撤、夏普比率等风险指标
- **收益分析**: 计算实际收益和费用收入